# The Rebalancing Bonus — a quantitative teardown 🔬
### Real multi-asset total-return tape · Booth-Fama identity · HAC + block-bootstrap · sign controls

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Controls risk?: Confirmed](https://img.shields.io/badge/Controls_risk%3F-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We separate the two things rebalancing does — **reduce variance** (a real, exact identity) and **add return vs drift** (not certifiable, regime-dependent) — and show the famous +0.9 pts/yr "diversification return" answers a *different question* than the investor's.

> ⚠️ **Not investment advice.** SPY + TLT daily, total-return adjusted (`quantlab.data`, Yahoo); aligned common window from TLT's 2002-07 inception; 10 bps one-way per rebalance. Sources in [`docs/references.md`](../docs/references.md), reproducible run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (free_rebalance/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from free_rebalance import data, strategy

AS_OF = "2026-06-12"
frame = data.load_real(("SPY", "TLT"), mode="total_return").loc[:AS_OF]
annual    = strategy.run_portfolio(frame, freq="annual",    cost_bps=10.0)
quarterly = strategy.run_portfolio(frame, freq="quarterly", cost_bps=10.0)
drift     = strategy.run_portfolio(frame, freq="drift",     cost_bps=10.0)
bonus_a = annual['cagr'] - drift['cagr']
print(f"basket SPY/TLT total return: {len(frame):,} rows  {frame.index[0].date()} -> {frame.index[-1].date()}  fp={data.fingerprint(frame)}")
print(f"rebalanced(A) CAGR {annual['cagr']*100:.2f}%  vol {annual['vol']*100:.1f}%  Sharpe {annual['sharpe']:.3f}")
print(f"drift (B&H)   CAGR {drift['cagr']*100:.2f}%  vol {drift['vol']*100:.1f}%  Sharpe {drift['sharpe']:.3f}")
print(f"realised rebalancing bonus (annual) = {bonus_a*100:+.2f} pts/yr CAGR")


basket SPY/TLT total return: 6,007 rows  2002-07-30 -> 2026-06-12  fp=f644f699f3af
rebalanced(A) CAGR 8.22%  vol 9.6%  Sharpe 0.875
drift (B&H)   CAGR 8.82%  vol 10.3%  Sharpe 0.874
realised rebalancing bonus (annual) = -0.59 pts/yr CAGR


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `WEAK` | Realised bonus over drift **-0.59 pts/yr** (annual); HAC *t* on the daily bonus = **-1.19**, block-bootstrap 95% CI **[-0.660, +0.132] bps/day** straddles 0. Flips sign by regime. |
| **Tradability** | `MIRAGE` | Trails drift by **0.6 pts/yr** (8.22% vs 8.82%); Sharpe a wash (0.875 vs 0.874). The +0.9 pts/yr Booth-Fama "bonus" is vs the weighted-avg of assets, not vs drift. |
| **Controls risk?** | `CONFIRMED` | Vol **9.6%** vs **10.3%**; the book stays at target weights instead of drifting. |

> 💡 **In plain words:** rebalancing is a *risk control that works* wearing the costume of a *return booster that doesn't*.

## 1 · The claim, steelmanned

- **H₁ (variance):** a fixed-weight rebalanced portfolio has lower variance than the weighted average of its constituents — the Booth-Fama diversification return, `DR = 0.5(Σ wᵢσᵢ² − σ_p²) ≥ 0`.
- **H₂ (the sold claim):** therefore rebalancing *adds return* over **buy-and-hold / drift** — a free lunch.

H₁ is an **exact identity** and always holds. H₂ does **not** follow, because the drift portfolio is a different (and also-diversified) baseline. That gap is the whole study.

## 2 · So what? — what rides on each answer

If H₂ held, the optimal policy would be *rebalance as often as costs allow*. It doesn't — and seeing *why* (the bonus's baseline isn't your benchmark; trends punish trimming) is the lesson.

## 3 · How we'd know — the protocol

Rebalanced vs drift on the real tape · the exact Booth-Fama identity · HAC *t* + circular block-bootstrap CI on the **daily bonus** · the sub-period sign · two synthetic sign controls (mean-revert → +, trend → −).

## 4 · The teardown

Headline stats, all net of 10 bps one-way per rebalance:

In [2]:
tbl = pd.DataFrame({
  'CAGR %':  [annual['cagr']*100, quarterly['cagr']*100, drift['cagr']*100],
  'Vol %':   [annual['vol']*100, quarterly['vol']*100, drift['vol']*100],
  'Sharpe':  [annual['sharpe'], quarterly['sharpe'], drift['sharpe']],
  'MaxDD %': [annual['max_dd']*100, quarterly['max_dd']*100, drift['max_dd']*100],
  'Rebal':   [annual['n_rebal'], quarterly['n_rebal'], drift['n_rebal']],
}, index=['Rebalanced (annual)', 'Rebalanced (quarterly)', 'Drift (B&H)'])
tbl.round(3)

,CAGR %,Vol %,Sharpe,MaxDD %,Rebal
Rebalanced (annual),8.225,9.555,0.875,-28.907,25
Rebalanced (quarterly),8.217,9.719,0.861,-29.022,96
Drift (B&H),8.817,10.274,0.874,-26.888,0


**The Booth-Fama / Willenbrock identity** — the textbook 'bonus', and its real baseline:

In [3]:
dr = strategy.diversification_return(frame)
print(f"diversification return DR = 0.5*(wavg_var - var_p) = {dr['dr_annual']*100:+.2f} pts/yr")
print(f"  weighted-avg variance {dr['wavg_var']*100:.2f}  -  portfolio variance {dr['var_p']*100:.2f}  =  var reduction {dr['var_reduction']*100:.2f} pts")
print(f"realised bonus vs DRIFT (annual)    = {(annual['cagr']-drift['cagr'])*100:+.2f} pts/yr")
print('=> the identity is >=0 by construction, but its baseline is the weighted-avg of assets,')
print('   NOT the drift portfolio -- which is why the honest bonus can be (and is) negative.')

diversification return DR = 0.5*(wavg_var - var_p) = +0.90 pts/yr
  weighted-avg variance 2.80  -  portfolio variance 0.99  =  var reduction 1.81 pts
realised bonus vs DRIFT (annual)    = -0.59 pts/yr
=> the identity is >=0 by construction, but its baseline is the weighted-avg of assets,
   NOT the drift portfolio -- which is why the honest bonus can be (and is) negative.


> 💡 **In plain words:** the +0.9 pts/yr is real — but it's the gap between rebalancing and an investor who already split the difference between the two assets *every day by hand*. Nobody does that. The real-world alternative is to **buy and hold**, and against *that* the bonus vanishes.

**HAC inference + block-bootstrap CI** on the daily bonus (annual rebalanced − drift):

In [4]:
diff = (annual['net'] - drift['net']).to_numpy()
t = strategy.hac_tstat(diff)
lo, hi = strategy.block_bootstrap_ci(diff, block=63, n_boot=2000)
print(f'mean daily bonus = {diff.mean()*1e4:+.3f} bps/day   HAC t = {t:+.2f}')
print(f'block-bootstrap 95% CI on mean daily bonus = [{lo*1e4:+.3f}, {hi*1e4:+.3f}] bps/day')
print('=> CI straddles 0: the daily bonus is not statistically distinguishable from zero.')

mean daily bonus = -0.245 bps/day   HAC t = -1.19
block-bootstrap 95% CI on mean daily bonus = [-0.660, +0.132] bps/day
=> CI straddles 0: the daily bonus is not statistically distinguishable from zero.


> 💡 **In plain words:** day to day, rebalanced and drift are the same bet within the noise. There is no reliably-signed edge to harvest.

**The bonus by sub-period** — the regime dependence, quantified:

In [5]:
yrs = frame.index.year
for mask, lab in [(yrs <= 2013, '2002-2013'), (yrs >= 2014, '2014-2026')]:
    sub = frame.loc[mask]
    ra = strategy.run_portfolio(sub, freq='annual', cost_bps=10.0)
    rd = strategy.run_portfolio(sub, freq='drift',  cost_bps=10.0)
    print(f'{lab}: realised bonus {(ra["cagr"]-rd["cagr"])*100:+.2f} pts/yr')

2002-2013: realised bonus +1.27 pts/yr


2014-2026: realised bonus -1.74 pts/yr


**The synthetic sign controls** — proof the harness reads the *sign* correctly:

In [6]:
for label, (fr, truth) in [('mean-revert (Shannon)', data.synthetic_meanrevert(seed=102)),
                           ('one asset trends',      data.synthetic_trend(seed=102))]:
    r = strategy.rebalancing_bonus(fr, freq='quarterly', cost_bps=10.0)
    exp = '+' if truth['expected_bonus_sign'] > 0 else '-'
    got = '+' if r['bonus_cagr'] >= 0 else '-'
    print(f"{label:24s} bonus {r['bonus_cagr']*100:+.2f} pts/yr  (got {got}, expected {exp})  {'OK' if got==exp else 'WRONG'}")

mean-revert (Shannon)    bonus +0.77 pts/yr  (got +, expected +)  OK


one asset trends         bonus -3.90 pts/yr  (got -, expected -)  OK


> 💡 **In plain words:** plant mean-reversion and the harness banks a positive bonus; plant a trend and it correctly reports a negative one. So the negative real-tape reading is a *finding about the regime*, not a broken measurement.

## 5 · The verdict

Signal `WEAK` (bonus -0.59 pts/yr, HAC *t* -1.19, CI straddles 0, sign flips by regime), Tradability `MIRAGE` for the free-lunch claim (−0.6 pts/yr vs drift, Sharpe a wash), Controls risk? `CONFIRMED` (vol 9.6% vs 10.3%). The +0.9 pts/yr diversification return is a variance identity vs the wrong baseline, not a bankable edge over buy-and-hold.

## 6 · Could you trade it?

Capacity is a non-issue (SPY/TLT). The binding fact is economic: there is **no signed return bonus** to capture over buy-and-hold, and higher rebalancing frequency only raises costs. The defensible reason to rebalance is **risk discipline** — keeping the book at its chosen risk level — which is worth doing, but it is not the advertised free lunch.

## 7 · Going further

- Add **GLD** and recompute the bonus on a 3-asset book — more low-correlation legs raise the Booth-Fama DR, but does the *realised* bonus over drift turn positive?
- **Threshold** (tolerance-band) rebalancing vs calendar — trade only on drift, compare cost-adjusted bonus.
- Condition the bonus on a **dispersion/mean-reversion** regime statistic and test the split formally (the sub-period flip suggests it's predictable in part).